# Day 069 — Exercise 5: Extract Pipeline

**What you'll build:** `extract_pipeline(img, schema_cls, describe_fn=None)` — the complete end-to-end pipeline from PIL Image to validated Pydantic model.

**Why it matters:** This is the final integration layer. It encodes the image, calls the vision LLM with retries, validates the result, and returns a typed Python object — or raises a clear `ValueError` at any failure point. The `ImageExtractor` class is a thin wrapper over this function.

In [ ]:
import io
import re
import json
import base64
from pydantic import BaseModel

def image_to_base64(img, format='PNG'):
    buf = io.BytesIO()
    out = img
    if format.upper() in ('JPEG', 'JPG') and img.mode in ('RGBA', 'P'):
        out = img.convert('RGB')
    out.save(buf, format=format)
    return base64.b64encode(buf.getvalue()).decode()

def build_extraction_prompt(schema_cls):
    schema_json = json.dumps(schema_cls.model_json_schema(), indent=2)
    return (
        'Extract structured data from this image and return ONLY valid JSON '
        'matching this schema exactly. Do not include any explanation, '
        'markdown, or code blocks.\n\n'
        f'Schema:\n{schema_json}\n\n'
        'Return ONLY the JSON object, nothing else.'
    )

def strip_json_from_response(response):
    block = re.search(r'```(?:json)?\s*([\s\S]*?)```', response)
    if block:
        return block.group(1).strip()
    obj = re.search(r'\{[\s\S]*\}', response)
    if obj:
        return obj.group(0).strip()
    raise ValueError(f'No JSON found: {response[:200]!r}')

def extract_from_image(img_b64, schema_cls, describe_fn=None):
    prompt = build_extraction_prompt(schema_cls)
    if describe_fn is not None:
        response = describe_fn(img_b64, prompt)
    else:
        import ollama
        resp = ollama.chat(
            model='llava',
            messages=[{'role': 'user', 'content': prompt, 'images': [img_b64]}]
        )
        response = resp['message']['content']
    raw = strip_json_from_response(response)
    return json.loads(raw)

def safe_extract(img_b64, schema_cls, describe_fn=None, retries=2):
    for attempt in range(retries + 1):
        try:
            return extract_from_image(img_b64, schema_cls, describe_fn=describe_fn)
        except (ValueError, json.JSONDecodeError):
            if attempt == retries:
                return None
    return None

def validate_extraction(data, schema_cls):
    try:
        model = schema_cls.model_validate(data)
        return (True, model)
    except Exception as exc:
        return (False, str(exc))

from pydantic import BaseModel

class _TestItem(BaseModel):
    name:  str
    value: float

class _TestSchema(BaseModel):
    title:   str
    amount:  float
    items:   list[_TestItem] = []
    note:    str = ''

from PIL import Image
_img = Image.new('RGB', (200, 100), 'white')


## Task

Implement `extract_pipeline(img, schema_cls, describe_fn=None)`:

1. `img_b64 = image_to_base64(img)`
2. `data = safe_extract(img_b64, schema_cls, describe_fn=describe_fn, retries=2)`
3. If `data is None`: `raise ValueError('Extraction failed after retries')`
4. `ok, result = validate_extraction(data, schema_cls)`
5. If `not ok`: `raise ValueError(result)`
6. `return result`

## Your Implementation

In [ ]:
def extract_pipeline(img, schema_cls, describe_fn=None):
    """Full extraction pipeline: PIL Image → validated Pydantic model.

    Steps:
        1. image_to_base64(img)
        2. safe_extract(img_b64, schema_cls, describe_fn, retries=2)
        3. If None: raise ValueError('Extraction failed after retries')
        4. validate_extraction(data, schema_cls)
        5. If not ok: raise ValueError(error_message)
        6. Return the validated model instance

    Args:
        img:         PIL Image
        schema_cls:  Pydantic model class
        describe_fn: callable(img_b64, prompt) -> str for testing
    Returns:
        Validated Pydantic model instance
    Raises:
        ValueError if extraction or validation fails
    """
    raise NotImplementedError


In [ ]:
def extract_pipeline(img, schema_cls, describe_fn=None):
    img_b64 = image_to_base64(img)
    data = safe_extract(img_b64, schema_cls, describe_fn=describe_fn, retries=2)
    if data is None:
        raise ValueError('Extraction failed after retries')
    ok, result = validate_extraction(data, schema_cls)
    if not ok:
        raise ValueError(result)
    return result


## Automated checks

In [ ]:
score, total = 0, 5
try:
    _mock = lambda b, p: '{"title": "Invoice", "amount": 99.50}'

    # Happy path: returns validated model
    result = extract_pipeline(_img, _TestSchema, describe_fn=_mock)
    assert hasattr(result, 'title') and hasattr(result, 'amount'), (
        f"Expected model with title/amount, got {type(result)}")
    score += 1; print("\u2705 returns a validated Pydantic model")

    assert result.title == 'Invoice' and abs(result.amount - 99.50) < 0.001
    score += 1; print(f"\u2705 model fields correct: title={result.title!r}, amount={result.amount}")

    # Extraction failure → ValueError
    raised = False
    try:
        extract_pipeline(_img, _TestSchema, describe_fn=lambda b, p: 'no json here')
    except ValueError:
        raised = True
    assert raised, "Should raise ValueError when extraction fails"
    score += 1; print("\u2705 raises ValueError when extraction fails")

    # Validation failure → ValueError (missing required 'amount' field)
    raised2 = False
    try:
        extract_pipeline(_img, _TestSchema,
                          describe_fn=lambda b, p: '{"title": "OnlyTitle"}')
    except ValueError:
        raised2 = True
    assert raised2, "Should raise ValueError when validation fails"
    score += 1; print("\u2705 raises ValueError when validation fails (missing required field)")

    # Type coercion: '99.50' string → float 99.50
    result2 = extract_pipeline(_img, _TestSchema,
                                describe_fn=lambda b, p: '{"title": "T", "amount": "99.50"}')
    assert abs(result2.amount - 99.50) < 0.001
    score += 1; print("\u2705 Pydantic coerces string amounts to float")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def extract_pipeline(img, schema_cls, describe_fn=None):
    img_b64 = image_to_base64(img)
    data = safe_extract(img_b64, schema_cls, describe_fn=describe_fn, retries=2)
    if data is None:
        raise ValueError('Extraction failed after retries')
    ok, result = validate_extraction(data, schema_cls)
    if not ok:
        raise ValueError(result)
    return result
```

**Why `raise ValueError(result)` when validation fails?** The `result` from `validate_extraction` on failure is the error message string. Passing it to `ValueError` means the caller's exception message includes Pydantic's description of what was wrong — which field is missing, which type was wrong. This is more useful than a generic 'validation failed' message.

</details>